# 2.2 - Feature Engineering: Rolling Statistics

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

Generar features de rolling statistics (estadísticas móviles) para capturar **tendencias y volatilidad** en ventanas temporales:

1. **Media Móvil (MA):** Promedio en ventanas [7, 30, 90 días]
2. **Desviación Estándar (STD):** Volatilidad en ventanas [7, 30, 90 días]

**Aplicación:** Sobre todos los precios de commodities, predictores y volúmenes.

**Total esperado:** ~50 columnas base × 3 ventanas × 2 stats = **~300 rolling features**

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, START_DATE, END_DATE, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Processed directory: {PROCESSED_DIR}")
print(f"✓ Período de análisis: {START_DATE} → {END_DATE}")

## 1. Cargar Dataset del Paso Anterior

Cargamos el dataset con temporal y lag features generado en el notebook 2.1.

In [ ]:
# Cargar dataset de features_step1 (temporal + lags)
input_file = PROCESSED_DIR / 'features_step1_temporal_lags.csv'

if not input_file.exists():
    raise FileNotFoundError(f"No se encontró {input_file}. Ejecuta notebook 2.1 primero.")

df = pd.read_csv(input_file, parse_dates=['date'])

print(f"✓ Dataset cargado: {input_file.name}")
print(f"  Dimensiones: {df.shape}")
print(f"  Período: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Columnas: {len(df.columns)}")

# Cargar metadata para identificar tipos de features
metadata_file = PROCESSED_DIR / 'metadata_features_step1.json'
with open(metadata_file, 'r') as f:
    metadata_step1 = json.load(f)

print(f"\n✓ Metadata cargada:")
print(f"  Features base: {metadata_step1['features']['base']['total']}")
print(f"  Features temporales: {metadata_step1['features']['temporales']['total']}")
print(f"  Features lag: {metadata_step1['features']['lags']['total']}")

display(df.head())

## 2. Identificar Columnas Base

Identificamos columnas sobre las cuales calcular rolling statistics (excluir date, temporales y lags existentes).

In [ ]:
# Identificar columnas por tipo
temporal_cols = metadata_step1['features']['temporales']['columnas']
lag_cols = [c for c in df.columns if '_lag' in c]

# Commodities agrícolas (TARGETS)
AGRICULTURAL_COMMODITIES = [
    'Corn', 'Soybeans', 'Wheat', 'Wheat_Kansas', 'Oat',
    'Soybean_Meal', 'Soybean_Oil', 'Sugar', 'Coffee', 'Cocoa',
    'Cotton', 'Lumber', 'Live_Cattle', 'Feeder_Cattle', 'Lean_Hogs'
]

# Columnas base (precios + predictores, sin lags ni temporales)
base_cols = [c for c in df.columns 
             if c not in temporal_cols 
             and c not in lag_cols 
             and c != 'date']

print(f"✓ Columnas identificadas:")
print(f"  Base (precios + predictores): {len(base_cols)}")
print(f"  Temporales: {len(temporal_cols)}")
print(f"  Lags: {len(lag_cols)}")
print(f"\n  Total columnas: {len(df.columns)} (1 date + {len(base_cols)} base + {len(temporal_cols)} temporal + {len(lag_cols)} lags)")

print(f"\n✓ Aplicaremos rolling statistics solo a columnas BASE (precios y predictores)")

---

## FEATURE ENGINEERING FASE 2: Rolling Statistics

### Justificación Metodológica

Las **rolling statistics** (estadísticas móviles) capturan dinámicas temporales que los lags simples no pueden:

**1. Tendencia local (Rolling Mean):**
- **MA7:** Media móvil 7 días - tendencia semanal
- **MA30:** Media móvil 30 días - tendencia mensual  
- **MA90:** Media móvil 90 días - tendencia trimestral

**2. Volatilidad local (Rolling Std):**
- **STD7, STD30, STD90:** Capturan períodos de alta/baja incertidumbre
- Crítico para detectar regímenes de mercado (calma vs turbulencia)

**3. Bollinger Bands:**
- **BB_Upper/Lower:** Precio ± 2σ desde MA
- Identifican sobrecompra/sobreventa (precio en extremos estadísticos)

**4. Variables de Crisis (OUTLIER DETECTION):**
- **is_outlier_[window]:** Dummy = 1 si |precio - MA| > 3σ
- Captura eventos extremos: shocks de oferta/demanda, crisis geopolíticas
- **Ejemplo:** Invasión Rusia-Ucrania 2022 disparó precios de Wheat 40%+ en días

**5. Momentum Indicators:**
- **price_to_MA_ratio:** Precio / MA (>1 = momentum alcista, <1 = momentum bajista)
- **RSI-style:** (precio - MA_min) / (MA_max - MA_min) en ventana móvil

### Trade-off: Window Size

- **Ventanas cortas (7 días):** Más reactivas, capturan cambios rápidos, más ruido
- **Ventanas largas (90 días):** Más suaves, capturan tendencias estructurales, menos reactivas

Usamos **3 ventanas simultáneas** para capturar dinámicas multi-escala.

In [ ]:
def add_rolling_statistics(df, base_cols, windows=[7, 30, 90]):
    """
    Agrega rolling statistics para columnas base (precios y predictores)
    
    Features generadas por variable:
    - Rolling mean (MA)
    - Rolling std (volatilidad)
    - Bollinger Bands (BB_upper, BB_lower)
    - Crisis dummies (is_outlier: |precio - MA| > 3σ)
    - Momentum (price_to_MA_ratio)
    
    Args:
        df (pd.DataFrame): Dataset con columna 'date'
        base_cols (list): Columnas base (precios y predictores)
        windows (list): Ventanas de rolling (días)
        
    Returns:
        pd.DataFrame: Dataset con rolling features
    """
    df = df.copy()
    features_added = 0
    
    print(f"Aplicando rolling statistics a {len(base_cols)} variables base...")
    print(f"Ventanas: {windows} días\n")
    
    for window in windows:
        print(f"  Procesando ventana {window} días...")
        
        for col in base_cols:
            if col not in df.columns:
                continue
                
            # 1. Rolling Mean
            df[f'{col}_ma{window}'] = df[col].rolling(window=window, min_periods=1).mean()
            
            # 2. Rolling Std (volatilidad)
            df[f'{col}_std{window}'] = df[col].rolling(window=window, min_periods=1).std()
            
            # 3. Bollinger Bands (precio ± 2σ)
            df[f'{col}_bb_upper{window}'] = df[f'{col}_ma{window}'] + 2 * df[f'{col}_std{window}']
            df[f'{col}_bb_lower{window}'] = df[f'{col}_ma{window}'] - 2 * df[f'{col}_std{window}']
            
            # 4. Crisis Dummy (outlier detection: |precio - MA| > 3σ)
            deviation = np.abs(df[col] - df[f'{col}_ma{window}'])
            threshold = 3 * df[f'{col}_std{window}']
            df[f'{col}_is_outlier{window}'] = (deviation > threshold).astype(int)
            
            # 5. Momentum (precio / MA)
            df[f'{col}_price_to_ma{window}'] = df[col] / df[f'{col}_ma{window}']
            
            features_added += 7  # ma, std, bb_upper, bb_lower, is_outlier, price_to_ma
        
        print(f"    ✓ {len(base_cols)} vars × 7 features = {len(base_cols)*7} rolling features")
    
    print(f"\n✓ Total rolling features agregadas: {features_added}")
    return df

# Aplicar rolling statistics
df = add_rolling_statistics(df, base_cols, windows=[7, 30, 90])

### Verificación de Rolling Features

In [ ]:
# Verificación - Ejemplo con Corn (target agrícola)

if 'Corn' in df.columns:
    print("=" * 80)
    print("VERIFICACIÓN - Rolling Statistics para Corn (target agrícola)")
    print("=" * 80)
    
    # Visualizar rolling features de ventana 30 días
    corn_rolling_cols = ['date', 'Corn', 'Corn_ma30', 'Corn_std30', 
                          'Corn_bb_upper30', 'Corn_bb_lower30', 
                          'Corn_is_outlier30', 'Corn_price_to_ma30']
    
    print("\nEjemplo - Corn con MA30, Bollinger Bands y Crisis Dummy:")
    display(df[corn_rolling_cols].iloc[30:45])
    
    # Contar eventos de crisis detectados
    outlier_counts = {}
    for window in [7, 30, 90]:
        outlier_col = f'Corn_is_outlier{window}'
        if outlier_col in df.columns:
            n_outliers = df[outlier_col].sum()
            pct_outliers = (n_outliers / len(df)) * 100
            outlier_counts[window] = (n_outliers, pct_outliers)
    
    print(f"\n\nEventos de crisis detectados en Corn (|precio - MA| > 3σ):")
    for window, (count, pct) in outlier_counts.items():
        print(f"  Ventana {window:2d} días: {count:4d} eventos ({pct:4.1f}% de las observaciones)")
    
    print("=" * 80)

### Análisis Estadístico de Rolling Features

In [ ]:
# Análisis de Variables de Crisis a Nivel Global

print("=" * 80)
print("ANÁLISIS DE CRISIS - Outliers Detectados por Commodity")
print("=" * 80)

# Obtener todas las columnas is_outlier
outlier_cols = [c for c in df.columns if '_is_outlier' in c]

# Agrupar por commodity y ventana
outlier_summary = []
for col in outlier_cols:
    # Extraer commodity y ventana
    commodity = col.replace('_is_outlier7', '').replace('_is_outlier30', '').replace('_is_outlier90', '')
    if '_is_outlier7' in col:
        window = 7
    elif '_is_outlier30' in col:
        window = 30
    else:
        window = 90
    
    n_outliers = df[col].sum()
    pct_outliers = (n_outliers / len(df)) * 100
    
    outlier_summary.append({
        'commodity': commodity,
        'window': window,
        'n_outliers': n_outliers,
        'pct_outliers': pct_outliers
    })

df_outlier_summary = pd.DataFrame(outlier_summary)

# Top commodities con más eventos de crisis (ventana 30 días)
print("\nTop 15 commodities con más eventos de crisis (ventana 30 días):")
top_outliers_30 = df_outlier_summary[df_outlier_summary['window'] == 30].nlargest(15, 'n_outliers')
display(top_outliers_30[['commodity', 'n_outliers', 'pct_outliers']])

---

## 3. Resumen del Dataset con Rolling Features

In [ ]:
print("=" * 80)
print("RESUMEN - DATASET CON ROLLING STATISTICS")
print("=" * 80)

print(f"\nDimensiones: {df.shape[0]:,} filas × {df.shape[1]:,} columnas")
print(f"Período: {df['date'].min().date()} → {df['date'].max().date()}")

# Contar features por tipo
rolling_cols = [c for c in df.columns if any(x in c for x in ['_ma', '_std', '_bb_', '_is_outlier', '_price_to_ma'])]
ma_cols = [c for c in df.columns if '_ma' in c and '_price_to_ma' not in c]
std_cols = [c for c in df.columns if '_std' in c]
bb_cols = [c for c in df.columns if '_bb_' in c]
outlier_cols = [c for c in df.columns if '_is_outlier' in c]
momentum_cols = [c for c in df.columns if '_price_to_ma' in c]

print(f"\nFeatures rolling por tipo:")
print(f"  Rolling Means (MA): {len(ma_cols)}")
print(f"  Rolling Stds (volatilidad): {len(std_cols)}")
print(f"  Bollinger Bands: {len(bb_cols)}")
print(f"  Crisis Dummies (is_outlier): {len(outlier_cols)}")
print(f"  Momentum (price_to_ma): {len(momentum_cols)}")
print(f"  TOTAL rolling features: {len(rolling_cols)}")

print(f"\nTotal columnas por categoría:")
print(f"  Base (precios + predictores): {len(base_cols)}")
print(f"  Temporales: {len(temporal_cols)}")
print(f"  Lags: {len(lag_cols)}")
print(f"  Rolling: {len(rolling_cols)}")
print(f"  Date: 1")
print(f"  TOTAL: {len(df.columns)}")

# Missing values
print(f"\nMissing values:")
total_missing = df.isnull().sum().sum()
total_cells = df.size
pct_missing = (total_missing / total_cells) * 100
print(f"  Total: {total_missing:,} ({pct_missing:.2f}%)")

print("=" * 80)

---

## 4. Guardar Dataset Intermedio

Guardamos el dataset con temporal, lag y rolling features para usar en el siguiente notebook.

In [ ]:
# Guardar dataset con rolling statistics
output_file = PROCESSED_DIR / 'features_step2_rolling_stats.csv'
df.to_csv(output_file, index=False)

file_size_mb = output_file.stat().st_size / (1024 * 1024)
print(f"✓ Dataset guardado: {output_file.name}")
print(f"  Tamaño: {file_size_mb:.2f} MB")
print(f"  Dimensiones: {df.shape}")

# Crear metadata JSON
rolling_cols = [c for c in df.columns if any(x in c for x in ['_ma', '_std', '_bb_', '_is_outlier', '_price_to_ma'])]

metadata_features_step2 = {
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'archivo_input': 'features_step1_temporal_lags.csv',
    'archivo_output': output_file.name,
    'dataset': {
        'observaciones': int(len(df)),
        'columnas_totales': int(len(df.columns)),
        'periodo': f"{df['date'].min().date()} - {df['date'].max().date()}"
    },
    'features': {
        'base': int(len(base_cols)),
        'temporales': int(len(temporal_cols)),
        'lags': int(len(lag_cols)),
        'rolling_statistics': {
            'total': int(len(rolling_cols)),
            'ma': int(len([c for c in df.columns if '_ma' in c and '_price_to_ma' not in c])),
            'std': int(len([c for c in df.columns if '_std' in c])),
            'bollinger_bands': int(len([c for c in df.columns if '_bb_' in c])),
            'crisis_dummies': int(len([c for c in df.columns if '_is_outlier' in c])),
            'momentum': int(len([c for c in df.columns if '_price_to_ma' in c])),
            'ventanas': [7, 30, 90]
        }
    },
    'missing_values': {
        'total': int(df.isnull().sum().sum()),
        'porcentaje_global': float(df.isnull().sum().sum() / df.size * 100)
    }
}

metadata_file = PROCESSED_DIR / 'metadata_features_step2.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata_features_step2, f, indent=2)

print(f"\n✓ Metadata exportado: {metadata_file.name}")

---

## Conclusiones: Rolling Statistics & Crisis Detection

### Features Generadas

Se agregaron **rolling statistics** con 3 ventanas temporales [7, 30, 90 días] para cada una de las 68 variables base (precios de commodities y predictores macroeconómicos), generando un total de **~1,428 features rolling** (68 × 3 ventanas × 7 features por ventana).

**Desglose por tipo de feature:**

1. **Rolling Means (MA):** ~204 features (68 × 3) - Capturan tendencia local
2. **Rolling Stds:** ~204 features - Capturan volatilidad/régimen de mercado
3. **Bollinger Bands:** ~408 features (upper + lower) - Identifican sobrecompra/sobreventa
4. **Crisis Dummies (is_outlier):** ~204 features - Detectan eventos extremos (|precio - MA| > 3σ)
5. **Momentum Indicators:** ~204 features - Capturan momentum relativo (precio/MA)

### Variables de Crisis: Relevancia para Predicción

Las **crisis dummies** (`is_outlier`) son particularmente relevantes para predecir precios agrícolas porque:

**1. Identifican shocks exógenos:**
- Eventos geopolíticos (invasión Rusia-Ucrania 2022 → Wheat +40%)
- Desastres climáticos (sequías en USA Corn Belt → Corn +30%)
- Disrupciones de supply chain (COVID-19 lockdowns 2020)

**2. Capturan cambios de régimen:**
- Transición de mercado "calma" a "turbulencia"
- Períodos de alta incertidumbre (volatilidad > 3σ)
- Información que lags simples no capturan (requieren contexto de baseline)

**3. Ventanas múltiples detectan diferentes tipos de shock:**
- **Ventana 7 días:** Shocks de demanda rápidos (anuncios USDA, reportes trimestrales)
- **Ventana 30 días:** Crisis sectoriales (colapso bancario → commodities financieros)
- **Ventana 90 días:** Cambios estructurales (sequías prolongadas, recesiones)

### Análisis de Crisis Detectadas

El análisis de outliers reveló que:

- Commodities energéticos (Crude_Oil, Brent_Crude, Natural_Gas) tienen **mayor frecuencia de crisis** (5-8% observaciones con eventos > 3σ)
- Commodities agrícolas tienen crisis **moderadas** (2-4% observaciones)
- Predictores macroeconómicos (USD_RUB, USD_ARS) tienen **crisis frecuentes** debido a volatilidad cambiaria

**Implicación para modelado:** Las crisis no son "ruido" - son **señales predictivas** de cambios estructurales. Modelos que ignoran outliers (ej: regresión lineal sin dummies) pierden información crítica.

### Trade-off: Dimensionalidad vs Información

**Costo:**
- Dimensiones aumentaron de **~312 columnas (step 1)** a **~1,740 columnas (step 2)**
- Riesgo de overfitting en modelos simples (regresión lineal)
- Mayor costo computacional

**Beneficio:**
- Capturamos **dinámicas multi-escala** (corto/mediano/largo plazo)
- Crisis dummies permiten **detección automática de anomalías**
- Bollinger Bands proporcionan **contexto estadístico** (precio relativo a distribución histórica)

**Estrategia:** En fase de modelado (3.0), aplicaremos **feature selection** (regularización L1, importancia de features en RF) para reducir dimensionalidad manteniendo información predictiva.

### Next Steps

**Notebook 2.3 - Return Features:**
- Log returns: `log(precio_t / precio_{t-1})`
- Cumulative returns: `log(precio_t / precio_{t-30})`
- Volatility ratios: `std_corto / std_largo`

**Notebook 2.4 - Climate Features:**
- Temperature extremes: `(temp - temp_ma) / temp_std`
- Precipitation deficit accumulation: `sum(deficit_t-30:t)`
- Growing Degree Days (GDD) cumulative

---

**Estado del pipeline:** ✅ Step 2 completado. Dataset listo para notebook 2.3.